# Airbnb Data — DVA Statistical Analysis
**Dataset:** `cleaned_airbnb.csv` | **Rows:** 32,585 | **Columns:** 36

---
### Contents
1. Setup & Data Loading
2. Descriptive Statistics (Mean, Median, Mode)
3. Hypothesis Testing (t-test, ANOVA, Chi-Square)
4. Correlation Analysis
5. Regression Analysis (Linear & Multiple)

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('cleaned_airbnb.csv')

# Clean price column (remove $ and commas if present)
df['price'] = df['price'].astype(str).str.replace('[$,]', '', regex=True).astype(float)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 2. Descriptive Statistics — Mean, Median, Mode

In [ ]:
# Select key numeric columns for descriptive analysis
numeric_cols = ['price', 'review_scores_rating', 'accommodates', 'bedrooms',
                'beds', 'bathrooms', 'availability_365', 'number_of_reviews',
                'reviews_per_month', 'estimated_revenue_l365d']

desc = df[numeric_cols].describe().T
desc['median'] = df[numeric_cols].median()
desc['mode']   = df[numeric_cols].mode().iloc[0]
desc['skew']   = df[numeric_cols].skew()

print('=== Descriptive Statistics ===')
desc[['mean', 'median', 'mode', 'std', 'min', 'max', 'skew']]

In [ ]:
# Distribution plots for key variables
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_cols = ['price', 'review_scores_rating', 'accommodates',
             'availability_365', 'reviews_per_month', 'estimated_revenue_l365d']

for ax, col in zip(axes.flatten(), plot_cols):
    data = df[col].dropna()
    # Cap outliers at 99th percentile for readability
    cap = data.quantile(0.99)
    data = data[data <= cap]
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean {data.mean():.1f}')
    ax.axvline(data.median(), color='green',  linestyle='--', linewidth=1.5, label=f'Median {data.median():.1f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Distributions with Mean & Median (capped at 99th pct)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mode for categorical columns
cat_cols = ['room_type', 'property_type', 'neighbourhood_cleansed']
print('=== Mode for Categorical Columns ===')
for col in cat_cols:
    mode_val = df[col].mode()[0]
    freq     = df[col].value_counts().iloc[0]
    pct      = freq / len(df) * 100
    print(f'{col:35s}: {mode_val!r:<30} (n={freq}, {pct:.1f}%)')

## 3. Hypothesis Testing

### 3a. Independent Samples t-test
**H₀:** Superhost listings and non-superhost listings have the same average price.  
**H₁:** They differ.

In [ ]:
superhost     = df[df['host_is_superhost'] == 1]['price'].dropna()
non_superhost = df[df['host_is_superhost'] == 0]['price'].dropna()

t_stat, p_val = stats.ttest_ind(superhost, non_superhost, equal_var=False)  # Welch's t-test

print('--- Independent t-test: Price ~ Superhost Status ---')
print(f'Superhost     — Mean: ${superhost.mean():.2f},     n={len(superhost)}')
print(f'Non-Superhost — Mean: ${non_superhost.mean():.2f}, n={len(non_superhost)}')
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_val:.4f}')
print()
if p_val < 0.05:
    print('✅ Reject H₀ — Significant price difference between superhost and non-superhost (α=0.05)')
else:
    print('❌ Fail to reject H₀ — No significant price difference at α=0.05')

# Box plot
fig, ax = plt.subplots()
price_cap = df['price'].quantile(0.95)
plot_df = df[df['price'] <= price_cap].copy()
plot_df['Superhost'] = plot_df['host_is_superhost'].map({1: 'Superhost', 0: 'Non-Superhost'})
sns.boxplot(data=plot_df, x='Superhost', y='price', palette='Set2', ax=ax)
ax.set_title(f't-test: Price vs Superhost Status\np-value = {p_val:.4f}')
ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

### 3b. One-Way ANOVA
**H₀:** Average price is the same across all room types.  
**H₁:** At least one room type has a different average price.

In [ ]:
room_groups = [grp['price'].dropna().values
               for _, grp in df.groupby('room_type')]

f_stat, p_val_anova = stats.f_oneway(*room_groups)

print('--- One-Way ANOVA: Price ~ Room Type ---')
print(df.groupby('room_type')['price'].agg(['mean', 'median', 'count']).round(2))
print()
print(f'F-statistic : {f_stat:.4f}')
print(f'p-value     : {p_val_anova:.4e}')
print()
if p_val_anova < 0.05:
    print('✅ Reject H₀ — Significant price difference across room types (α=0.05)')
else:
    print('❌ Fail to reject H₀')

fig, ax = plt.subplots()
sns.boxplot(data=df[df['price'] <= price_cap], x='room_type', y='price', palette='pastel', ax=ax)
ax.set_title(f'ANOVA: Price by Room Type\nF={f_stat:.2f}, p={p_val_anova:.2e}')
ax.set_xlabel('Room Type')
ax.set_ylabel('Price (USD)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 3c. Chi-Square Test of Independence
**H₀:** Room type and instant bookable status are independent.  
**H₁:** They are associated.

In [ ]:
contingency = pd.crosstab(df['room_type'], df['instant_bookable'])
contingency.columns = ['Not Instant', 'Instant Bookable']
print('Contingency Table:')
print(contingency)
print()

chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-Square  : {chi2:.4f}')
print(f'p-value     : {p_chi2:.4e}')
print(f'Degrees of freedom: {dof}')
print()
if p_chi2 < 0.05:
    print('✅ Reject H₀ — Room type and instant bookable are significantly associated')
else:
    print('❌ Fail to reject H₀')

contingency.plot(kind='bar', stacked=True, colormap='Set2', figsize=(9, 5))
plt.title(f'Chi-Square: Room Type vs Instant Bookable\np-value = {p_chi2:.4e}')
plt.xlabel('Room Type')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
corr_cols = ['price', 'review_scores_rating', 'review_scores_accuracy',
             'review_scores_cleanliness', 'review_scores_location',
             'review_scores_value', 'accommodates', 'bedrooms', 'beds',
             'bathrooms', 'availability_365', 'number_of_reviews',
             'reviews_per_month', 'estimated_revenue_l365d',
             'host_acceptance_rate', 'minimum_nights']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size': 7}, ax=ax)
ax.set_title('Pearson Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with price
price_corr = corr_matrix['price'].drop('price').sort_values(key=abs, ascending=False)
print('=== Top Correlations with Price ===')
print(price_corr.round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['steelblue' if v > 0 else 'tomato' for v in price_corr]
price_corr.plot(kind='barh', color=colors, ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with Price')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

In [ ]:
# Spearman correlation (robust to outliers / non-normal distributions)
spearman_corr = df[corr_cols].corr(method='spearman')
spearman_price = spearman_corr['price'].drop('price').sort_values(key=abs, ascending=False)
print('=== Spearman Correlation with Price ===')
print(spearman_price.round(4).to_string())

In [ ]:
# Scatter plots for top 4 correlated numeric features vs price
top4 = price_corr.abs().nlargest(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, col in zip(axes.flatten(), top4):
    sub = df[[col, 'price']].dropna()
    cap_p = sub['price'].quantile(0.95)
    cap_c = sub[col].quantile(0.99)
    sub = sub[(sub['price'] <= cap_p) & (sub[col] <= cap_c)]
    r, p = stats.pearsonr(sub[col], sub['price'])
    ax.scatter(sub[col], sub['price'], alpha=0.15, s=8, color='steelblue')
    m, b = np.polyfit(sub[col], sub['price'], 1)
    x_line = np.linspace(sub[col].min(), sub[col].max(), 200)
    ax.plot(x_line, m * x_line + b, 'r-', linewidth=2)
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Price (USD)')
    ax.set_title(f'{col.replace("_", " ").title()} vs Price\nr = {r:.3f}, p = {p:.3e}')

plt.suptitle('Top Correlated Features vs Price', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Regression Analysis

### 5a. Simple Linear Regression — Price ~ Accommodates

In [ ]:
from scipy.stats import linregress

slr_data = df[['accommodates', 'price']].dropna()
slr_data = slr_data[slr_data['price'] <= slr_data['price'].quantile(0.95)]

X_slr = slr_data['accommodates'].values
y_slr = slr_data['price'].values

slope, intercept, r_value, p_value, std_err = linregress(X_slr, y_slr)

print('=== Simple Linear Regression: Price ~ Accommodates ===')
print(f'Intercept  : {intercept:.4f}')
print(f'Slope      : {slope:.4f}  (price increases ${slope:.2f} per additional guest)')
print(f'R²         : {r_value**2:.4f}')
print(f'p-value    : {p_value:.4e}')
print(f'Std Error  : {std_err:.4f}')

fig, ax = plt.subplots()
ax.scatter(X_slr, y_slr, alpha=0.1, s=10, color='steelblue', label='Data')
x_line = np.linspace(X_slr.min(), X_slr.max(), 200)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2.5, label=f'y = {slope:.1f}x + {intercept:.1f}')
ax.set_xlabel('Accommodates (guests)')
ax.set_ylabel('Price (USD)')
ax.set_title(f'Simple Linear Regression\nR² = {r_value**2:.3f}, p = {p_value:.2e}')
ax.legend()
plt.tight_layout()
plt.show()

### 5b. Multiple Linear Regression — Price ~ Multiple Features

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ['accommodates', 'bedrooms', 'beds', 'bathrooms',
            'availability_365', 'number_of_reviews', 'reviews_per_month',
            'host_acceptance_rate', 'minimum_nights',
            'review_scores_rating', 'host_is_superhost', 'instant_bookable']

mlr_df = df[features + ['price']].dropna()
mlr_df = mlr_df[mlr_df['price'] <= mlr_df['price'].quantile(0.95)]

X = mlr_df[features]
y = mlr_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = np.mean(np.abs(y_test.values - y_pred))

print('=== Multiple Linear Regression: Price ~ Features ===')
print(f'R²   (test): {r2:.4f}')
print(f'RMSE (test): {rmse:.4f}')
print(f'MAE  (test): {mae:.4f}')
print()
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

In [ ]:
# Coefficient plot
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Multiple Regression — Feature Coefficients (Standardised)\nR² = {r2:.3f}')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.15, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'Actual vs Predicted\nR² = {r2:.3f}')

residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.15, s=8, color='darkorange')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('Residual Distribution')
ax.set_xlabel('Residual (Actual - Predicted)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

# Shapiro-Wilk on a sample (full residuals too large)
sample_residuals = residuals[:5000]
stat, p_norm = stats.shapiro(sample_residuals)
print(f'Shapiro-Wilk test on residuals (n=5000): stat={stat:.4f}, p={p_norm:.4e}')
if p_norm < 0.05:
    print('Residuals are NOT normally distributed — consider log-transform of price or robust regression.')
else:
    print('Residuals appear normally distributed.')

---
## Summary Table

In [ ]:
summary = pd.DataFrame([
    ['Mean Price',           f'${df["price"].mean():.2f}'],
    ['Median Price',         f'${df["price"].median():.2f}'],
    ['Mode Price',           f'${df["price"].mode()[0]:.2f}'],
    ['t-test p-value',       f'{p_val:.4f} — Superhost vs Non-Superhost'],
    ['ANOVA p-value',        f'{p_val_anova:.4e} — Price across Room Types'],
    ['Chi-Square p-value',   f'{p_chi2:.4e} — Room Type vs Instant Bookable'],
    ['Top Pearson Corr',     f'{price_corr.index[0]} (r={price_corr.iloc[0]:.3f})'],
    ['Simple LR R²',         f'{r_value**2:.4f} (Price ~ Accommodates)'],
    ['Multiple LR R²',       f'{r2:.4f}'],
    ['Multiple LR RMSE',     f'{rmse:.2f}'],
], columns=['Metric', 'Value'])

print(summary.to_string(index=False))